In [36]:
import pyautogui as pyg
import time

pyg.FAILSAFE = True

print("2秒後開始執行，請切換至 Chrome 恐龍遊戲視窗...")
pyg.sleep(2)
pyg.press('space') # 自動按一下啟動遊戲

# 使用你最順手的精準掃描範圍
scan_region = (420, 580, 160, 110)

last_jump_time = 0
jump_duration = 0.35   # 恐龍完整滯空時間（秒）
buffered_jump = False  # 紀錄落地後的待命跳躍緩衝

while True:
    im = pyg.screenshot(region=scan_region)
    
    # 【核心優化】動態抓取當前畫面的背景色（取左上角天空像素）
    # 這樣在日夜交替轉場的瞬間，背景顏色改變了，程式也能即時同步，完全不會有反應不及的問題！
    bg_color = im.getpixel((10, 10))
    
    action = None # 'jump' 或 'duck' 或 None
    
    # -------------------------------------------------------------
    # 1. 優先檢查【地面層】（仙人掌 / 低空障礙）：起跳動作
    # 透過計算像素與當下背景的「色彩差異」，日夜通用、轉場秒懂！
    # -------------------------------------------------------------
    for x in range(15, 140):
        for y in range(65, 95): # 地面掃描範圍
            c = im.getpixel((x, y))
            # 計算 RGB 三原色的總色差值
            diff = abs(c[0] - bg_color[0]) + abs(c[1] - bg_color[1]) + abs(c[2] - bg_color[2])
            if diff > 100: # 只要跟背景有明顯色差，就是障礙物
                action = 'jump'
                break
        if action:
            break
                
    # -------------------------------------------------------------
    # 2. 其次檢查【空中層】（中空鳥需要低頭，高空鳥保持不變）
    # -------------------------------------------------------------
    if not action:
        for x in range(15, 140):
            for y in range(15, 38): # 中空鳥高度（避開地面仙人掌頂端）
                c = im.getpixel((x, y))
                diff = abs(c[0] - bg_color[0]) + abs(c[1] - bg_color[1]) + abs(c[2] - bg_color[2])
                if diff > 100:
                    action = 'duck'
                    break
            if action:
                break
                    
    # -------------------------------------------------------------
    # 3. 執行動作與「跳躍緩衝（Jump Buffer）」機制
    # -------------------------------------------------------------
    current_time = time.time()
    is_in_air = (current_time - last_jump_time < jump_duration)
    
    if action == 'jump':
        if is_in_air:
            # 如果在空中時遇到下一個連續仙人掌，把起跳指令緩衝起來
            buffered_jump = True
        else:
            # 在地面則直接起跳
            pyg.press('space')
            last_jump_time = time.time()
            buffered_jump = False
            
    # 一落地立刻執行緩衝的跳躍（完美解決連續仙人掌問題）
    if buffered_jump and not is_in_air:
        pyg.press('space')
        last_jump_time = time.time()
        buffered_jump = False

    # 執行低頭（中空鳥）
    if action == 'duck' and not is_in_air:
        pyg.keyDown('down')
        time.sleep(0.32)
        pyg.keyUp('down')
        
    time.sleep(0.01)

2秒後開始執行，請切換至 Chrome 恐龍遊戲視窗...


KeyboardInterrupt: 